# HELEN — Big-Model Fine-Tune (Colab A100)

Train the HELEN models your 5070 can't hold: **Qwen3.5-9B** (faithful, helen-core's size) or **Gemma 4 12B** (multimodal), on the same governed 148-example dataset.

Runtime: **Runtime → Change runtime type → A100 GPU** (40 GB). 9B bf16 LoRA ≈ 22 GB; 12B 4-bit LoRA ≈ 18 GB — both fit.

**Governance:** the LoRA adapter this produces is a `WEIGHT_UPDATE` claim, not a verdict. Receipt it, validate in a fresh context (proposer ≠ validator), then *you* authorize promotion. The verifier is not sovereign; you are.

In [ ]:
# 0. Confirm we got an A100 (40 GB). If this shows T4/L4, switch the runtime.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1. Install Unsloth (pulls transformers v5, required for Qwen3.5).
!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

In [ ]:
# 2. Pick the model. Flip this one line.
MODEL_CHOICE = "qwen35-9b"   # "qwen35-9b" (faithful) | "gemma-12b" (multimodal)
MAX_SEQ = 4096

CFG = {
    "qwen35-9b": dict(model="unsloth/Qwen3.5-9B", family="qwen",
                      load_in_4bit=False, load_in_16bit=True,
                      instr="<|im_start|>user\n", resp="<|im_start|>assistant\n"),
    "gemma-12b": dict(model="unsloth/gemma-4-12b-it", family="gemma",
                      load_in_4bit=True, load_in_16bit=False,
                      instr="<|turn>user\n", resp="<|turn>model\n"),
}[MODEL_CHOICE]
print("Training:", CFG["model"])

In [ ]:
# 3. Upload the 3 HELEN dataset files (from helen_gemma_finetune/ in the repo):
#    helen_persona_sft.jsonl  helen_plugins_sft.jsonl  helen_doctrine_sft.jsonl
from google.colab import files
up = files.upload()
DATA = [f for f in up if f.endswith(".jsonl")]
print("datasets:", DATA)

In [ ]:
# 4. Load the base model + attach LoRA (branches per family).
if CFG["family"] == "qwen":
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CFG["model"], max_seq_length=MAX_SEQ,
        load_in_4bit=False, load_in_16bit=True, full_finetuning=False)
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth", random_state=3407, max_seq_length=MAX_SEQ)
else:  # gemma
    from unsloth import FastModel
    from unsloth.chat_templates import get_chat_template
    model, tokenizer = FastModel.from_pretrained(
        model_name=CFG["model"], max_seq_length=MAX_SEQ,
        load_in_4bit=True, full_finetuning=False)
    tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")  # non-thinking: train on final answers
    model = FastModel.get_peft_model(
        model, finetune_vision_layers=False, finetune_language_layers=True,
        finetune_attention_modules=True, finetune_mlp_modules=True,
        r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407)

In [ ]:
# 5. Format the HELEN dataset through the model's chat template.
from datasets import load_dataset
ds = load_dataset("json", data_files=DATA, split="train")
def fmt(ex):
    return {"text": [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
                     for c in ex["conversations"]]}
ds = ds.map(fmt, batched=True)
print(ds[0]["text"][:300])

In [ ]:
# 6. Train (HELEN's assistant turns only). E2B/E4B loss 13-15 is normal; 9B/12B run lower.
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(max_seq_length=MAX_SEQ, dataset_text_field="text",
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        logging_steps=1, optim="adamw_8bit", weight_decay=0.001,
        lr_scheduler_type="linear", seed=3407, report_to="none",
        output_dir="outputs_helen_big"))
trainer = train_on_responses_only(trainer, instruction_part=CFG["instr"], response_part=CFG["resp"])
trainer.train()

In [ ]:
# 7. Export to GGUF (q4_k_m) and download for Ollama.
import hashlib, glob, json
model.save_pretrained_gguf("helen_big_gguf", tokenizer, quantization_method="q4_k_m")
g = glob.glob("helen_big_gguf/*.gguf")[0]
sha = hashlib.sha256(open(g,'rb').read()).hexdigest()[:16]
print("WEIGHT_UPDATE receipt:", {"model": CFG["model"], "datasets": DATA, "gguf": g, "render_sha256": sha})
from google.colab import files as _f; _f.download(g)

## Load the result into Ollama (back on your machine)
```bash
# put the downloaded .gguf next to a Modelfile:
#   FROM ./helen-big.gguf
#   PARAMETER temperature 0.4   (qwen)  /  1.0  (gemma)
ollama create helen-big -f Modelfile
ollama run helen-big "who are you?"
```
Then serve to the Mac via the 5070's Ollama, or run wherever you keep the model.

**Governance close-out:** the `render_sha256` printed in cell 7 is the adapter/GGUF receipt. Record it, validate the model against held-out HELEN behaviour in a fresh context, then authorize promotion. Proposer ≠ validator — even for HELEN's own weights.